### Import Dependencies

In [5]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree

In [6]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

### Embedding function

In [7]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small"
    }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.data[0].embedding

### Retrieval function

In [8]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [9]:
@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

### Format retrieved context function

In [10]:
@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

### Create prompt template function

In [11]:
@traceable(
    name="build_prompt",
    run_type="prompt"
)
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

### Generate answer function

In [12]:
@traceable(
    name="generate_answer",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-nano"
    }
)
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.choices[0].message.content

### Combined RAG pipeline

In [13]:
@traceable(
    name="rag_pipeline",
)
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer
    

In [14]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have a few USB-connectable fans that work well for hot summers:

1) Marame 120mm 5v USB Powered Fan with Speed Controller (ID: B0BRJS644Z)
- USB powered (3.3 ft cable)
- Speed controller on the cord: off / low / medium / high
- Built for cooling small electronics like router/modem/DVR/Xbox TV boxes, etc.

2) HZD Mini Portable USB Fan (ID: B0BXC72RLD)
- USB powered (4.9 ft cable)
- 3 speeds (low/medium/high)
- Quiet operation (noise listed as less than 50d)

If you tell me whether you want a fan for a desk/personal use or for cooling electronics (router/TV box), I can recommend the best match.


In [15]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Sure. Here are earphones from the available products with a rating above 4.0:

1) B0CH8DRD6K (rating 4.9)
Wired 3.5mm iPhone/Android earphones with microphone and volume control (also supports Siri). Ergonomic semi-in-ear design.

2) B0BRV544MV (rating 4.4)
Wireless Bluetooth 5.3 earbuds with 40H playtime (case + earbuds), deep bass, IPX7 waterproof, and 4-mic clear call.

3) B0CFHWF326 (rating 4.1)
Bluetooth 5.3 sports headband earphones with built-in ENC mic, designed for workouts/calling. 20+ hours playtime, skin-friendly breathable fabric.

4) B0BM657X74 (rating 4.6)
Bluetooth sleep headband (sleep mask + headphones style) with built-in mic and volume control; designed for side sleepers. 10–12H playtime; washable headband.

If you tell me whether you want wired or wireless (and whether for sports or sleep), I can narrow it down further.


In [16]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

Here are earphones from the available products with a rating below 4.0:

1) pamu S29 Wireless Earbuds Active Noise Cancelling (B09TFM1SFQ) - Rating 3.9
- Bluetooth 5.2 with automatic pairing
- ANC/ENC noise cancellation and 4-mic call noise reduction
- Battery: about 6 hours per charge + 24 hours in case

2) RUNAR RNR1 Wireless Neckband Running Headphones (B0BC4PGXFK) - Rating 3.8
- Bluetooth 5.0 neckband design
- Sweatproof/rainproof; compatible with TF/SD memory cards
- Collapsible, lightweight design; no noise cancellation
